# Barter-RS: Market Data Streaming

This notebook demonstrates how to use `barter-data` to stream real-time market data from cryptocurrency exchanges.

**Prerequisites**: Install the [evcxr Jupyter kernel](https://github.com/evcxr/evcxr/blob/main/evcxr_jupyter/README.md) for Rust.

## Topics Covered
- Streaming public trades from Binance Futures
- Streaming L1 order book data (best bid/ask)
- Streaming L2 order book data with local book management
- Error handling and reconnection

## Setup

First, add the required dependencies. Since evcxr resolves crates from crates.io, we reference the published versions.

In [ ]:
:dep barter-data = { version = "0.11" }
:dep barter-instrument = { version = "0.3" }
:dep tokio = { version = "1", features = ["full"] }
:dep futures-util = "0.3"
:dep tracing = "0.1"
:dep tracing-subscriber = { version = "0.3", features = ["env-filter"] }

## 1. Streaming Public Trades

The `Streams` builder provides a fluent API for subscribing to market data.
Each `.subscribe()` call creates a **separate WebSocket connection**, which is useful
for isolating high-volume instruments.

In [ ]:
use barter_data::{
    exchange::binance::futures::BinanceFuturesUsd,
    streams::{Streams, reconnect::stream::ReconnectingStream},
    subscription::trade::PublicTrades,
};
use barter_instrument::instrument::market_data::kind::MarketDataInstrumentKind;
use futures_util::StreamExt;

// Build streams for BTC and ETH perpetual futures on Binance
let streams = Streams::<PublicTrades>::builder()
    // High-volume: dedicated WebSocket for BTC
    .subscribe([
        (BinanceFuturesUsd::default(), "btc", "usdt", MarketDataInstrumentKind::Perpetual, PublicTrades),
    ])
    // High-volume: dedicated WebSocket for ETH
    .subscribe([
        (BinanceFuturesUsd::default(), "eth", "usdt", MarketDataInstrumentKind::Perpetual, PublicTrades),
    ])
    // Lower volume instruments can share a single connection
    .subscribe([
        (BinanceFuturesUsd::default(), "sol", "usdt", MarketDataInstrumentKind::Perpetual, PublicTrades),
        (BinanceFuturesUsd::default(), "avax", "usdt", MarketDataInstrumentKind::Perpetual, PublicTrades),
    ])
    .init()
    .await
    .expect("failed to initialise streams");

println!("Streams initialised! Reading first 5 trades...");

// Merge all exchange streams and consume events
let mut joined = streams
    .select_all()
    .with_error_handler(|error| eprintln!("Stream error: {error:?}"));

let mut count = 0;
while let Some(event) = joined.next().await {
    println!(
        "[{}] {} {} | price={} qty={}",
        event.exchange, event.instrument, event.kind.side,
        event.kind.price, event.kind.quantity
    );
    count += 1;
    if count >= 5 { break; }
}

## 2. Streaming L1 Order Books (Best Bid/Ask)

L1 order book data provides the best bid and ask prices — useful for spread monitoring
and simple execution logic.

In [ ]:
use barter_data::{
    exchange::binance::spot::BinanceSpot,
    streams::{Streams, reconnect::stream::ReconnectingStream},
    subscription::book::OrderBooksL1,
};
use barter_instrument::instrument::market_data::kind::MarketDataInstrumentKind;
use futures_util::StreamExt;

let streams = Streams::<OrderBooksL1>::builder()
    .subscribe([
        (BinanceSpot::default(), "btc", "usdt", MarketDataInstrumentKind::Spot, OrderBooksL1),
        (BinanceSpot::default(), "eth", "usdt", MarketDataInstrumentKind::Spot, OrderBooksL1),
    ])
    .init()
    .await
    .expect("failed to initialise L1 streams");

println!("L1 OrderBook streams initialised! Reading first 5 updates...");

let mut joined = streams
    .select_all()
    .with_error_handler(|error| eprintln!("Stream error: {error:?}"));

let mut count = 0;
while let Some(event) = joined.next().await {
    println!(
        "[{}] {} | best_bid={} best_ask={}",
        event.exchange, event.instrument,
        event.kind.best_bid.price, event.kind.best_ask.price
    );
    count += 1;
    if count >= 5 { break; }
}

## 3. L2 Order Book Manager

For deeper market analysis, `barter-data` provides a managed L2 order book that
automatically applies incremental updates. The `OrderBookMap` gives thread-safe
read access to the latest snapshot at any time.

In [ ]:
use barter_data::{
    books::{manager::init_multi_order_book_l2_manager, map::OrderBookMap},
    exchange::binance::spot::BinanceSpot,
    subscription::book::OrderBooksL2,
};
use barter_instrument::instrument::market_data::{
    MarketDataInstrument, kind::MarketDataInstrumentKind,
};
use std::time::Duration;

// Initialise L2 OrderBook manager with separate connections per instrument group
let book_manager = init_multi_order_book_l2_manager([
    vec![
        (BinanceSpot::default(), "btc", "usdt", MarketDataInstrumentKind::Spot, OrderBooksL2),
    ],
    vec![
        (BinanceSpot::default(), "eth", "usdt", MarketDataInstrumentKind::Spot, OrderBooksL2),
    ],
]).await.expect("failed to init L2 book manager");

// Clone the book map for concurrent read access
let books = book_manager.books.clone();

// Spawn the manager to apply incremental updates in the background
tokio::spawn(book_manager.run());

// Wait for some updates to arrive, then read snapshots
let instrument = MarketDataInstrument::new("btc", "usdt", MarketDataInstrumentKind::Spot);

tokio::time::sleep(Duration::from_secs(3)).await;
if let Some(book) = books.find(&instrument) {
    let snapshot = book.read();
    println!("BTC/USDT L2 OrderBook snapshot:");
    println!("  Bids (top 3): {:?}", snapshot.bids.iter().take(3).collect::<Vec<_>>());
    println!("  Asks (top 3): {:?}", snapshot.asks.iter().take(3).collect::<Vec<_>>());
} else {
    println!("Book not yet available");
}

## Key Concepts

| Concept | Description |
|---------|-------------|
| `Streams::builder()` | Fluent API entry point. Each `.subscribe()` opens a new WebSocket. |
| `ReconnectingStream` | Wraps streams with automatic reconnection on transient errors. |
| `select_all()` | Merges all exchange streams into a single unified stream. |
| `with_error_handler()` | Attaches an error callback without terminating the stream. |
| `OrderBookMap` | Thread-safe (`Arc<RwLock>`) local order book with snapshot reads. |

### Subscription Types

- `PublicTrades` — individual trade executions
- `OrderBooksL1` — best bid/ask (top of book)
- `OrderBooksL2` — full depth order book with incremental updates